# Load the model that will generate qa pairs

In [1]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
    device_map="auto",  # Auto-map to GPU
)

# Enable native 2x faster inference
model = FastLanguageModel.for_inference(model)

print("Loaded model in 4-bit ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-06 19:59:49 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.11: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+5146f2a.d20251002. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded model in 4-bit ✅


# Load the corpus

In [2]:
import os 

#CORPUS_DIR = "/storage/corpus/wtk_archive_with_stops"
CORPUS_DIR = "/storage/corpus/ai_corpus_slimmer_clean"

BLOCK_SIZE = 1024  # max tokens per chunk

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": "</s>"})
    added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    added = True
if added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 2) Load raw text files (no EOS strings here)
# -----------------------
def load_txt_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not filename.endswith(".txt"):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read().strip()
            if txt:
                texts.append(txt)
    return texts

raw_texts = load_txt_corpus(CORPUS_DIR)
print(f"Loaded {len(raw_texts)} files from {CORPUS_DIR} ✅")

Loaded 11882 files from /storage/corpus/ai_corpus_slimmer_clean ✅


# Build the training chunks

In [3]:
from typing import List, Optional

def split_by_eos_or_chunk(
    text: str,
    tokenizer,
    min_tokens: int = 20,
    max_segments: Optional[int] = None,
    approx_token_budget: Optional[int] = 25_000,
    alt_markers: Optional[List[str]] = None,
) -> List[str]:
    """
    Prefer splitting on tokenizer's EOS token (string or id).
    If no EOS markers are present, fall back to token-length chunking.
    Ensures segments have at least `min_tokens` and stops around desired volume.
    """
    alt_markers = alt_markers or ["</s>", "<|im_end|>", "<|endoftext|>"]

    # 1) Try string-based split using known markers (fast path)
    s = text
    for m in alt_markers:
        if m and m in s:
            parts = [p.strip() for p in s.split(m)]
            # filter by token count, not chars
            kept = []
            for p in parts:
                if not p:
                    continue
                if len(tokenizer.encode(p, add_special_tokens=False)) >= min_tokens:
                    kept.append(p)
            if kept:
                # enforce target volume and segment cap
                if approx_token_budget:
                    total = 0
                    budgeted = []
                    for seg in kept:
                        n = len(tokenizer.encode(seg, add_special_tokens=False))
                        if total + n > approx_token_budget or (max_segments and len(budgeted) >= max_segments):
                            break
                        budgeted.append(seg)
                        total += n
                    return budgeted
                if max_segments:
                    return kept[:max_segments]
                else:
                    return kept

    # 2) Fallback: chunk by tokens to ~fixed size if no markers found
    ids = tokenizer.encode(text, add_special_tokens=False)
    # choose a chunk size that’s friendly for your model/context
    CHUNK_TOKENS = max(min_tokens * 4, 1024)  # e.g., 512–1024
    segments = []
    i = 0
    total = 0
    while i < len(ids) and (approx_token_budget is None or total < approx_token_budget) and (not max_segments or len(segments) < max_segments):
        j = min(i + CHUNK_TOKENS, len(ids))
        seg_text = tokenizer.decode(ids[i:j], skip_special_tokens=True).strip()
        if seg_text:
            segments.append(seg_text)
            total += (j - i)
        i = j
    return segments

blocks = []
for text in raw_texts:
    segments = split_by_eos_or_chunk(
        text,
        tokenizer,
        min_tokens=50,
        max_segments=50,
        approx_token_budget=25_000,           # aim for ~25k tokens total
        alt_markers=["</s>", "<|im_end|>"],   # add any markers your data uses
    )
    blocks.extend(segments)
    if len(blocks) % 1000 == 0:
        print(f"Processed {len(blocks)} number of blocks...")
    #print(f"Procssed raw text with len {len(text)} which created len segments {len(segments)}")

print(f"Prepared {len(blocks)} packed training chunks ✅")


Processed 6000 number of blocks...
Processed 9000 number of blocks...
Processed 10000 number of blocks...
Processed 12000 number of blocks...
Processed 13000 number of blocks...
Processed 15000 number of blocks...
Processed 16000 number of blocks...
Processed 18000 number of blocks...
Processed 21000 number of blocks...
Processed 24000 number of blocks...
Prepared 25493 packed training chunks ✅


# Generate qa pairs (singular)

In [ ]:
SEGMENTS_LIMIT = 100000
MAX_TOKENS = 512
REPORTING_INTERVAL = 2
JSON_WRITE_INTERVAL = 100

qa_pairs = []

import json
import re

print(f"Generating qa pairs for {len(blocks)} number of blocks...")
      
i = 0
total_pairs = 0
for segment in blocks:
    i = i + 1
    if i > SEGMENTS_LIMIT:
        print(f"Hit segments limit {SEGMENTS_LIMIT}. Stopping.")
        break

    if i % REPORTING_INTERVAL == 0:
        print(f"Processed {i} number of segments...")

    # Truncate to ~512-1024 tokens for context length (align with max_seq_length=1024)
    tokens = segment.split()[:1024]
    segment = " ".join(tokens)

    # Prompt with escaped curly braces for JSON example
    prompt = f"""From this text segment: {segment}

Generate 5 diverse Q-A pairs:
- 2 factual (e.g., who/what/when/where).
- 1 explanatory (e.g., why/how).
- 1 inference-based (e.g., what might happen next?).
- 1 domain-specific (e.g., if code, explain function; if news, key implications).
Format as JSON list: [{{ "question": "...", "answer": "...</s>"}}, ...]"""

    
    system_prompt = f"""Ensure questions are specific, non-generic, and answers are concise  (<250 words) but include enough detail for the user to understand 
    the critical points, directly extracted/inferred from the segment.
    Do not output thinking /no_think"""
    
    try:        
        # Build messages
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt},
        ]

        has_chat_template = getattr(tokenizer, "chat_template", None) not in (None, "")

        # Encode
        if has_chat_template:
            enc = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            # enc can be a Tensor or a dict; normalize
            if isinstance(enc, torch.Tensor):
                inputs = {"input_ids": enc.to(model.device)}
                inputs["attention_mask"] = torch.ones_like(inputs["input_ids"])
            else:
                inputs = {k: v.to(model.device) for k, v in enc.items()}
                if "attention_mask" not in inputs:
                    inputs["attention_mask"] = torch.ones_like(inputs["input_ids"])
        else:
            prompt_str = (
                f"### System:\n{system_prompt}\n\n"
                f"### User:\n{user_prompt}\n\n"
                f"### Assistant:\n"
            )
            enc = tokenizer(prompt_str, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in enc.items()}

        # Tokenize and generate
        #inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True).to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,  # Limit for speed
            do_sample=False,  # Deterministic for consistency
            num_return_sequences=1,
            temperature=0.5,
            top_p=0.95,                          # limit selection to this set of cumulative probability              
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        #print(f"Response: {response}")

        # Parse JSON (trim prompt and handle </s>)
        think_start = response.find("[")
        json_start = response.find("[", think_start + 1)
        json_end = response.rfind("]", json_start + 1) + 1

        if json_start != -1 and json_end != -1:

            snippet = response[json_start:json_end]
            #print(f"SNIPPET {i}: {snippet}")
            
            qa_list = json.loads(snippet)

            # Ensure </s> in answers
            for qa in qa_list:
                if not qa["answer"].endswith("</s>"):
                    qa["answer"] += "</s>"
                qa_pairs.append(qa)
            
            total_pairs = total_pairs + 1
            if total_pairs % REPORTING_INTERVAL == 0:
                print(f"Processed {total_pairs} number of segments...")
                
    except Exception as e:
        print(f"Error processing segment {i}: {e}")
        continue

     # Save Q-A pairs to JSON for LoRA training on interval
    if i > 0 and i % JSON_WRITE_INTERVAL  == 0:
        output_filename = "qa_pairs_" + str(i) + ".json"
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
        print(f"Wrote {len(qa_pairs)} qa pairs to file {output_filename}...")
        qa_pairs = []
        
    # Clear GPU memory to prevent OOM
    torch.cuda.empty_cache()

# Save remaining Q-A pairs
if qa_pairs:
    output_filename = f"qa_pairs_{i}.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
    print(f"Wrote {len(qa_pairs)} QA pairs to file {output_filename}...")
    
print(f"Generated {len(total_pairs)} total QA pairs")

Generating qa pairs for 25493 number of blocks...
Processed 2 number of segments...
Processed 2 number of segments...
Processed 4 number of segments...
Processed 6 number of segments...
Processed 8 number of segments...
Processed 10 number of segments...
Processed 12 number of segments...
Processed 4 number of segments...
Processed 14 number of segments...
Processed 6 number of segments...
Processed 16 number of segments...
Processed 8 number of segments...
Processed 18 number of segments...
Processed 10 number of segments...
Processed 20 number of segments...
Processed 12 number of segments...
Processed 22 number of segments...
Processed 24 number of segments...
Processed 26 number of segments...
Processed 28 number of segments...
Processed 14 number of segments...
Processed 30 number of segments...
Processed 16 number of segments...


Unsloth: Input IDs of shape torch.Size([1, 1187]) with length 1187 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


Processed 32 number of segments...
Processed 34 number of segments...
Processed 18 number of segments...
Processed 36 number of segments...
Processed 38 number of segments...
Processed 40 number of segments...
Processed 42 number of segments...
Processed 20 number of segments...
Processed 44 number of segments...
Processed 46 number of segments...
Processed 22 number of segments...
Processed 48 number of segments...
Error processing segment 48: Extra data: line 1 column 120 (char 119)
Processed 24 number of segments...
Processed 50 number of segments...
Processed 52 number of segments...


Unsloth: Input IDs of shape torch.Size([1, 1103]) with length 1103 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


Processed 54 number of segments...
Processed 26 number of segments...
Processed 56 number of segments...


...

Processed 66 number of segments...
Processed 30 number of segments...
Processed 68 number of segments...
Processed 70 number of segments...
Processed 72 number of segments...
Processed 74 number of segments...
Processed 32 number of segments...
Processed 76 number of segments...
Processed 34 number of segments...
Processed 78 number of segments...
Error processing segment 79: Expecting value: line 1 column 1 (char 0)
Processed 80 number of segments...
Processed 36 number of segments...
Processed 82 number of segments...
Processed 38 number of segments...
Processed 84 number of segments...
Processed 86 number of segments...
Processed 88 number of segments...
Processed 90 number of segments...
Error processing segment 90: Expecting ':' delimiter: line 1 column 1317 (char 1316)
Processed 92 number of segments...
Processed 40 number of segments...
Processed 94 number of segments...
Processed 96 number of segments...
Processed 42 number of segments...
Error processing segment 97: Extra data

Unsloth: Input IDs of shape torch.Size([1, 1182]) with length 1182 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


Processed 138 number of segments...
Processed 140 number of segments...
Processed 142 number of segments...
Processed 144 number of segments...
Processed 146 number of segments...
Error processing segment 146: 'answer'
Processed 148 number of segments...
Processed 150 number of segments...
Processed 152 number of segments...
Processed 154 number of segments...
Processed 156 number of segments...
Processed 158 number of segments...
Processed 160 number of segments...
Processed 162 number of segments...
Processed 164 number of segments...
Processed 166 number of segments...
Processed 168 number of segments...
Processed 170 number of segments...
Processed 172 number of segments...
Error processing segment 172: 'answer'
Processed 66 number of segments...
Processed 174 number of segments...
Processed 68 number of segments...
Processed 176 number of segments...
Processed 70 number of segments...
Processed 178 number of segments...
Processed 180 number of segments...
Processed 182 number of s

Unsloth: Input IDs of shape torch.Size([1, 1133]) with length 1133 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


Processed 78 number of segments...
Processed 204 number of segments...
Processed 80 number of segments...
Processed 206 number of segments...
Processed 82 number of segments...
Processed 208 number of segments...
Processed 210 number of segments...
Processed 84 number of segments...
Processed 212 number of segments...
Processed 86 number of segments...
Processed 214 number of segments...
Error processing segment 215: Expecting ':' delimiter: line 4 column 95 (char 1136)
Processed 216 number of segments...
Processed 88 number of segments...
Processed 218 number of segments...


# Generate qa pairs (batched)

In [16]:
SEGMENTS_LIMIT = 100000
MAX_TOKENS = 300
REPORTING_INTERVAL = 10
JSON_WRITE_INTERVAL = 500
BATCH_SIZE = 8

import json
import torch
import nltk
from unsloth import FastLanguageModel
from nltk.tokenize import sent_tokenize

# Pre-tokenize prompts
prompts = []
for segment in blocks:
    # Truncate to ~512-1024 tokens (align with max_seq_length=1024)
    tokens = segment.split()[:1024]
    segment = " ".join(tokens)
    prompt = f"""From this text segment: {segment}

Generate 5 diverse Q-A pairs:
- 2 factual (e.g., who/what/when/where).
- 1 explanatory (e.g., why/how).
- 1 inference-based (e.g., what might happen next?).
- 1 domain-specific (e.g., if code, explain function; if news, key implications).
Ensure questions are specific, non-generic, and answers are concise (<250 words), directly extracted/inferred from the segment.
Include </s> at the end of each answer to match corpus format. /no_think
Format as JSON list: [{{ "question": "...", "answer": "...</s>"}}, ...]"""
    prompts.append(prompt)

qa_pairs = []
total_pairs = 0
i = 0

# Process in batches
for batch_start in range(0, len(prompts), BATCH_SIZE):
    if i >= SEGMENTS_LIMIT:
        print(f"Hit segments limit {SEGMENTS_LIMIT}. Stopping.")
        break

    batch_prompts = prompts[batch_start:batch_start + BATCH_SIZE]
    batch_size_actual = len(batch_prompts)
    i += batch_size_actual

    if i // BATCH_SIZE % REPORTING_INTERVAL == 0:
        print(f"Processed {i} number of segments...")

    try:
        # Batch tokenize
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            max_length=1024,
            truncation=True,
            padding=True,  # Pad for batch processing
        ).to("cuda")

        # Batch generate
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        # Decode batch outputs
        batch_responses = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Parse JSON from each response
        for j, response in enumerate(batch_responses):
            print(f"Response: {response}")
            json_start = response.find("[")
            json_end = response.rfind("]", json_start + 1) + 1
            print(f"Post-Response: {response[json_start:json_end]}")
            if json_start != -1 and json_end != -1:
                try:
                    qa_list = json.loads(response[json_start:json_end])
                    for qa in qa_list:
                        if not qa["answer"].endswith("</s>"):
                            qa["answer"] += "</s>"
                        qa_pairs.append(qa)
                        total_pairs += 1
                except json.JSONDecodeError:
                    print(f"Error parsing JSON for segment {batch_start + j + 1}")
                    print(f"PROBLEM: {response[json_start:json_end]}")
            else:
                print(f"Error: No JSON found in response for segment {batch_start + j + 1}")

    except Exception as e:
        print(f"Error processing batch {batch_start // batch_size + 1}: {e}")
        continue

    # Save Q-A pairs to JSON on interval
    if i > 0 and i % JSON_WRITE_INTERVAL == 0:
        output_filename = f"qa_pairs_{i}.json"
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
        print(f"Wrote {len(qa_pairs)} QA pairs to file {output_filename}...")
        qa_pairs = []

    # Clear GPU memory to prevent OOM
    torch.cuda.empty_cache()

# Save remaining Q-A pairs
if qa_pairs:
    output_filename = f"qa_pairs_{i}.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
    print(f"Wrote {len(qa_pairs)} QA pairs to file {output_filename}...")

print(f"Generated {total_pairs} total QA pairs")

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Response: From this text segment: Everything is natural and tastes so good microfarms push back against food apartheid Valérie Macon AFP Getty Images The Guardian One of the UK s Leading Newspapers June 10 2023 Posted June 18th 2023 In South Los Angeles Crop Swap LA volunteers and staffers harvested bags of freshly picked produce from the front yard of a residence Everything we re growing is nutrient dense and the food remains in the neighborhood says Jamiah Hargins who founded Crop Swap LA in 2018 as a small monthly swap of surplus produce After spending years in finance and consulting Hargins decided to create a local food distribution system to address the fact that his neighborhood was a food desert meaning most residents have little access to healthy food It s now one of many Bipoc led groups across the US that are reclaiming their agricultural heritage and redefining the local food movement by growing on traditional farms and unconventional spaces such as yards medians and vacant

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Response: From this text segment: rates HIV infection plummeted from an all time high in 2000 of 104 2 new cases per million to 4 2 cases per million in 2015 Portugals remarkable recovery could not have happened without an enormous cultural shift and a change in how the country viewed drugs Portugals policy rests on three pillars one that theres no such thing as a soft or hard drug only healthy and unhealthy relationships with drugs two that an individuals unhealthy relationship with drugs often conceals frayed relationships with loved ones with the world around them and with themselves and three that the eradication of all drugs is an impossible goal In spite of Portugals tangible results other countries have been reluctant to follow Note Portugal s successful policy has contributed to public health outcomes that starkly contrast US trends 2017 11 26 CNBC News Associated Press Posted 2017 12 11 04 03 30 Against the backdrop of the nation s largest Veterans Day parade Democratic Gov An

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


...

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Response: From this text segment: k and have conversations with his family after the exercises opened up his brain He s never been very musical so when Sue Ryder first suggested music therapy he said What good is that going to do I m a typical Northern man and I thought What s a girl with a guitar going to do for me get me to the gym But it really worked Clare sat me down and explained the process I learned that music is very unlike other therapies as it opens up all of the brain Note Watch a profoundly touching documentary about a man who takes on the broken healthcare system to demonstrate music s ability to heal combat memory loss and awaken the soul and the deepest parts of humanity Explore a treasure trove of concise summaries of incredibly inspiring news articles which will inspire you to make a difference 2023 03 10 Fortune No one chooses medical debt Many Americans who fall ill have no choice but to rack up debt in order to stay healthy or in some cases stay alive For the under

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Response: From this text segment: Literally a miracle Violent rival gangs in South Africa call truce to help people during pandemic CBS News April 18 2020 Posted May 11th 2020 Warring gangs in South Africa are working together in an unprecedented truce to deliver much needed food to people under lockdown The country has seen a 75 decrease in violent crime since it imposed strict restrictions over the coronavirus pandemic and normally dangerous streets in Cape Town now see sworn enemies meeting up to collect essential goods to distribute throughout hungry communities What we re seeing happen here is literally a miracle Pastor Andie Steele Smith said Steel Smith works with gang members in his community many of whom are convicted killers They are the best distributors in the country he said They are used to distributing other white powders but still they are distributing things and then they know everybody Preston Jacobs a member of the Americans gang told CBS News Debora Patta it feels n

KeyboardInterrupt: 